# Full Pipeline: GPAW to Resource Estimates

This notebook demonstrates the **complete end-to-end workflow** for quantum resource estimation of a periodic material:

1. **GPAW DFT** --- run the density functional theory calculation
2. **PawExtractor** --- extract PAW ingredients and save to HDF5
3. **PawReader + OneNormCalculator** --- load data (lazily) and compute $\lambda$
4. **ResourceEstimator** --- estimate Toffoli gates and logical qubits

We use metallic hydrogen (H fcc, $a = 3.67$ A) as the test system.

See notebook **02_gpaw_extraction** for detailed parameter explanations and notebook **04_one_norm_calculation** for the step-by-step $\lambda$ computation.

In [1]:
import warnings
import math
import numpy as np
from numpy.exceptions import ComplexWarning

# GPAW triggers harmless RuntimeWarnings (divide by zero, overflow in dot/matmul)
# on Apple Silicon due to numerical edge cases in the PAW setup and LFC phases.
# These do not affect results.
warnings.filterwarnings("ignore", category=RuntimeWarning, module="gpaw")
warnings.filterwarnings("ignore", category=RuntimeWarning, module="numpy")
warnings.filterwarnings("ignore", category=ComplexWarning)

## Step 1: GPAW DFT Calculation

Run a converged DFT calculation using GPAW. Symmetry must be turned off so the k-mesh contains all k-points explicitly (required by the quantum algorithm).

In [2]:
# Requires GPAW to be installed
from ase.build import bulk
from gpaw import GPAW

atoms = bulk('H', 'fcc', a=3.67)
calc = GPAW(
    mode="lcao", basis='dzp', h=0.30,
    kpts={'size': (2, 2, 2), 'gamma': True},
    xc='PBE', nbands=5,
    symmetry={'point_group': False, 'time_reversal': False}
)
atoms.calc = calc
energy = calc.get_potential_energy(atoms)
print(f"Total energy: {energy:.6f} eV")


  ___ ___ ___ _ _ _  
 |   |   |_  | | | | 
 | | | | | . | | | | 
 |__ |  _|___|_____|  25.7.0
 |___|_|             

User:   rbhardwaj@pn2503601.lanl.gov
Date:   Fri Feb 27 14:01:52 2026
Arch:   arm64
Pid:    60468
CWD:    /Users/rbhardwaj/Desktop/Fe Simulation/Code/Bloch-PAW-main/examples
Python: 3.12.10
gpaw:   /Users/rbhardwaj/.julia/conda/3/aarch64/lib/python3.12/site-packages/gpaw
_gpaw:  /Users/rbhardwaj/.julia/conda/3/aarch64/lib/python3.12/site-packages/
        _gpaw.cpython-312-darwin.so
ase:    /Users/rbhardwaj/.julia/conda/3/aarch64/lib/python3.12/site-packages/ase (version 3.26.0)
numpy:  /Users/rbhardwaj/.julia/conda/3/aarch64/lib/python3.12/site-packages/numpy (version 2.3.0)
scipy:  /Users/rbhardwaj/.julia/conda/3/aarch64/lib/python3.12/site-packages/scipy (version 1.15.3)
libxc:  2.x.y
units:  Angstrom and eV
cores: 1
OpenMP: False
OMP_NUM_THREADS: 1

Input parameters:
  basis: dzp
  h: 0.3
  kpts: {gamma: True,
         size: (2, 2, 2)}
  mode: lcao
  nbands: 5
  sy

## Step 2: Extract PAW Ingredients to HDF5

Extract the smooth pseudo pair-density, PAW tensors, and one-body matrix elements. We skip the expensive two-body $\kappa$ tensor (`write_two_body=False`) --- see notebook **02** for the accuracy trade-off discussion.

In [15]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent   # .../Bloch-PAW-main
sys.path.insert(0, str(repo_root))

In [ ]:
outpath = Path("../data/lcbo_2x2x2.h5")   # relative to examples/
outpath.parent.mkdir(parents=True, exist_ok=True)

hdf5_file = extractor.export_hdf5(
    filepath=str(outpath),
    write_two_body=False
)
print("HDF5 file written:", hdf5_file)

HDF5 file written: ../data/lcbo_2x2x2.h5


In [12]:
# Requires GPAW to be installed
from bloch_paw.extractor import PawExtractor

extractor = PawExtractor(calc, nbands=5,
    thr_rho=1e-3, thr_D=1e-2, thr_C=1e-2,
    thr_h=1e-5, thr_kappa=1e-5)

hdf5_file = extractor.export_hdf5(
    filepath="../data/lcbo_2x2x2.h5",
    write_two_body=False
)
print(f"HDF5 file written: {hdf5_file}")

HDF5 file written: ../data/lcbo_2x2x2.h5


## Step 3: Load Data and Compute $\lambda$

Use lazy loading to keep memory usage low. The pair density $\tilde{\rho}$ stays on disk and is read one $(\mathbf{k}, \mathbf{k}')$ block at a time.

In [13]:
from bloch_paw import PawReader, OneNormCalculator, ResourceEstimator

DATA_FILE = "../data/lcbo_2x2x2.h5"

reader = PawReader(DATA_FILE)
with reader:
    inputs = reader.to_calculator_inputs(lazy=True)
    one_norm_calc = OneNormCalculator(**inputs, thr_rank=3e-5,
                                     sv_floor=1e-12, scale_floor=1e-12)
    lam = one_norm_calc.lambda_one_norm()
    R_avg, R0 = one_norm_calc.compute_average_rank()

print(f"One-norm \u03bb = {lam:.4f}")
print(f"Average rank R_avg = {R_avg:.2f}")
print(f"One-body rank R0 = {R0:.0f}")

One-norm λ = 110.3538
Average rank R_avg = 56.12
One-body rank R0 = 40


## Step 4: Resource Estimation

Compute the Toffoli gate count and logical qubit count. We use chemical accuracy ($\varepsilon_\text{chem} = 1.6 \times 10^{-3}$ Ha $\approx$ 1 kcal/mol) divided by 5 as the QPE precision, reflecting a 5-way error budget split between QPE phase estimation, Hamiltonian truncation, finite basis set, finite k-mesh sampling, and coefficient loading (finite bit-precision for LCU amplitudes).

In [14]:
eps_chem = 27e-3       # chemical accuracy in Hartree (1 kcal/mol)
eps_qpe = eps_chem / 5  # QPE share of the error budget

est = ResourceEstimator.from_hdf5(DATA_FILE)

toffolis = est.toffoli_count_per_be(Rl=R_avg, R0=R0)
qubits = est.total_qubits(Rl=R_avg, R0=R0, lam=lam, eps_qpe=eps_qpe)
iters = math.ceil(math.pi * lam / eps_qpe)

print("=" * 60)
print("FULL PIPELINE RESULTS")
print("=" * 60)
print(f"System:              H fcc (a=3.67 A)")
print(f"K-mesh:              2x2x2 ({est.Nk} k-points)")
print(f"Bands:               {est.Nb}")
print(f"Plane waves:         {est.Npw}")
print(f"PAW pairs:           {est.P}")
print(f"LCU labels L:        {est.L:,}")
print()
print(f"One-norm \u03bb:          {lam:,.4f}")
print(f"Average rank R_avg:  {R_avg:.2f}")
print(f"One-body rank R0:    {R0:.0f}")
print()
print(f"QPE precision:       \u03b5_QPE = {eps_qpe:.1e}")
print(f"QPE iterations:      {iters:,}")
print()
print(f"Toffoli per query:   {toffolis:,}")
print(f"Total Toffoli:       {toffolis * iters:,}")
print(f"Logical qubits:      {qubits:,}")
print("=" * 60)

FULL PIPELINE RESULTS
System:              H fcc (a=3.67 A)
K-mesh:              2x2x2 (8 k-points)
Bands:               5
Plane waves:         511
PAW pairs:           15
LCU labels L:        8,416

One-norm λ:          110.3538
Average rank R_avg:  56.12
One-body rank R0:    40

QPE precision:       ε_QPE = 5.4e-03
QPE iterations:      64,202

Toffoli per query:   62,456
Total Toffoli:       4,009,800,112
Logical qubits:      10,234


## Resource Scaling with Supercell Size

The resource estimates above are for a single unit cell ($N_a = 1$ atom). In practice, materials with multiple atoms per unit cell or supercell calculations scale significantly. The `supercell_size` parameter in `export_hdf5()` records the intended supercell dimensions $(L_x, L_y, L_z)$.

### How Resources Scale

For a supercell with $N_a = L_x \times L_y \times L_z$ atoms relative to the primitive cell:

| Quantity | Scaling | Physical reason |
|----------|---------|----------------|
| Bands $N_b$ | $\times N_a$ | One band per electron per unit cell |
| Plane waves $N_\text{pw}$ | $\times N_a$ | Larger real-space cell = denser G-grid |
| One-norm $\lambda$ | $\sim N_a^2$ | Pairwise Coulomb interactions |
| Logical qubits | $\sim N_a^{1.5}$ | System register + ancillae |
| Toffoli gates | $\sim N_a^{3.5}$ | QROAM table size + query count |

### Concrete Example

Suppose your base calculation uses a 1x1x1 unit cell with a 3x3x3 k-mesh (27 k-points). Scaling to a 2x2x2 supercell means:

- $N_a = 8$ atoms (8x more than primitive cell)
- $N_b$ increases by 8x (8x more bands)
- $N_\text{pw}$ increases by ~8x (denser reciprocal grid)
- $\lambda \sim 64\times$ larger
- Qubits $\sim 23\times$ more
- Toffoli gates $\sim 2900\times$ more

This steep scaling is why choosing the minimal system size that captures the relevant physics is critical for practical quantum resource estimates.

To specify a supercell size during extraction:

```python
# Record that this is a 2x2x2 supercell for resource scaling
extractor.export_hdf5("supercell.h5", supercell_size=(2, 2, 2))
```